[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUHK-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/nb/STAT3009_Side.ipynb)

# Recommendation with Side Information

**STAT3009 · Recommender Systems**

> **Today:** use continuous, categorical, and optional text features to recommend for users and items with no interaction history. The model comparison is the point, not the absolute RMSE.

**Route:** setup (3 min) → side features (5 min) → cold-start split (8 min) → feature-only baseline (8 min) → ID-only MF (10 min) → two-tower (15 min) → optional text (10 min) → exercises.

## 1. Setup

We use the canteen dataset from the EDA/MF notebook. Every user has a synthetic food preference, and every dish has price, category, and simple flags. The ratings are synthetic, so compare models rather than chasing an absolute score.

In [1]:
import warnings

warnings.filterwarnings("ignore", message="The given NumPy array is not writable")

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from torch import nn

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

torch.manual_seed(0)
np.random.seed(0)

BASE = "https://raw.githubusercontent.com/statmlben/CUHK-STAT3009/main/dataset/cuhk-food"
ratings = pd.read_csv(f"{BASE}/ratings.csv")
users = pd.read_csv(f"{BASE}/users.csv")
dishes = pd.read_csv(f"{BASE}/dishes.csv")

df = ratings.merge(users, on="user_id").merge(dishes, on="dish_id")
print("rows:", len(df))
print("users:", df.user_id.nunique(), "dishes:", df.dish_id.nunique())
display(df.head())

rows: 12683
users: 320 dishes: 183


,user_id,dish_id,rating,split,food_preference,dish,canteen_id,canteen,price,category,spicy,fried,healthy
0,0,61,4.5000,test,Noodles,豉油皇飯,3,CUHK Central Canteen,25.9000,Rice,0,0,0
1,0,102,4.5000,train,Noodles,豬扒扁意粉,5,Pommerenke Student Centre Coffee Shop,37.7000,Noodles,0,0,0
2,0,42,3.0000,train,Noodles,黑胡椒煙燻火腿田園沙律,1,United College Student Canteen,36.0000,Salad,0,0,1
3,0,55,4.5000,test,Noodles,蔥油雞扒腸仔飯,1,United College Student Canteen,43.0000,Rice,0,0,0
4,0,138,4.5000,test,Noodles,精選雜菜雞蛋湯烏冬,7,Orchid Lodge Restaurant,36.0000,Noodles,0,0,1


## 2. What side information do we have?

The four feature groups from the lecture are all present:

- **continuous:** price, spicy, fried, healthy
- **categorical:** food preference, dish category, canteen
- **user ID and item ID:** the usual MF rows
- **optional text:** the dish name (we will use a simple text encoder later)

The useful features are known before the rating. Rating means and counts are **not** used here, because they are derived from the target and are unavailable for a new user or dish.

In [2]:
cat_user = ["food_preference"]
cat_item = ["category", "canteen_id"]
num_item = ["price", "spicy", "fried", "healthy"]

print("user preferences:", sorted(df.food_preference.unique()))
print("dish categories:", sorted(df.category.unique()))
print("canteens:", df.canteen_id.nunique())
print("numeric columns:", num_item)

display(df[["user_id", "food_preference", "dish", "price", "category", "spicy", "fried", "healthy", "rating"]].head())

user preferences: ['Burgers & Sandwiches', 'Cafe & Dessert', 'Noodles', 'Rice', 'Snacks', 'Vegetarian & lighter']
dish categories: ['Burgers & Sandwiches', 'Cafe & Dessert', 'Noodles', 'Others', 'Rice', 'Salad', 'Snacks', 'Vegetarian']
canteens: 8
numeric columns: ['price', 'spicy', 'fried', 'healthy']


,user_id,food_preference,dish,price,category,spicy,fried,healthy,rating
0,0,Noodles,豉油皇飯,25.9000,Rice,0,0,0,4.5000
1,0,Noodles,豬扒扁意粉,37.7000,Noodles,0,0,0,4.5000
2,0,Noodles,黑胡椒煙燻火腿田園沙律,36.0000,Salad,0,0,1,3.0000
3,0,Noodles,蔥油雞扒腸仔飯,43.0000,Rice,0,0,0,4.5000
4,0,Noodles,精選雜菜雞蛋湯烏冬,36.0000,Noodles,0,0,1,4.5000


## 3. Cold-start split

A random row split is not enough: every user still appears in training. For a real cold-start test, hold out **20% of users entirely**. Those users have no training rows, so their ID embedding cannot be learned.

We will also compare with a random split to show why the distinction matters.

In [3]:
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
all_users = np.array(sorted(df.user_id.unique()))
rng.shuffle(all_users)
n_cold = int(0.2 * len(all_users))
cold_users = set(all_users[:n_cold])

cold_train = df[~df.user_id.isin(cold_users)].copy()
cold_test = df[df.user_id.isin(cold_users)].copy()

print("cold users:", len(cold_users))
print("cold-train rows:", len(cold_train), "cold-test rows:", len(cold_test))
print("unseen test dishes:", (~cold_test.dish_id.isin(cold_train.dish_id)).sum())

random_train, random_test = train_test_split(df, test_size=0.2, random_state=0)
print("random test users unseen in train:", (~random_test.user_id.isin(random_train.user_id)).sum())
print("random test dishes unseen in train:", (~random_test.dish_id.isin(random_train.dish_id)).sum())

cold users: 64
cold-train rows: 10278 cold-test rows: 2405
unseen test dishes: 0
random test users unseen in train: 0
random test dishes unseen in train: 0


The random split has **zero** unseen users and zero unseen dishes. The cold split has 64 unseen users. A model can look excellent on the first split and still fail in production when a new user arrives.

## 4. Feature-only baseline

Encode the categorical features with one-hot encoding, standardize the numeric columns, and fit a Ridge regression. This model never sees a user ID or a dish ID. It works for a completely new user as long as the profile features are available.

In [4]:
user_enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(cold_train[cat_user])
item_enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(cold_train[cat_item])
num_scaler = StandardScaler().fit(cold_train[num_item])

def side_matrix(frame):
    Xu = user_enc.transform(frame[cat_user])
    Xi = np.column_stack([
        item_enc.transform(frame[cat_item]),
        num_scaler.transform(frame[num_item]),
    ])
    return np.column_stack([Xu, Xi])

X_train_side = side_matrix(cold_train)
X_test_side = side_matrix(cold_test)
y_train_cold = cold_train.rating.to_numpy()
y_test_cold = cold_test.rating.to_numpy()

ridge = Ridge(alpha=1.0).fit(X_train_side, y_train_cold)
pred_ridge = ridge.predict(X_test_side)

print("feature-only RMSE:", f"{root_mean_squared_error(y_test_cold, pred_ridge):.4f}")
print("feature-only MAE: ", f"{mean_absolute_error(y_test_cold, pred_ridge):.4f}")
print("global mean RMSE:", f"{root_mean_squared_error(y_test_cold, np.full_like(y_test_cold, y_train_cold.mean())):.4f}")

feature-only RMSE: 0.7558
feature-only MAE:  0.5997
global mean RMSE: 0.9113


The feature-only model already beats the global mean. That is the first piece of evidence that the side features carry real signal. But it cannot represent “this specific user likes this specific dish” beyond what the features say.

## 5. ID-only MF

Now fit the familiar biased MF model. We reuse the ALS estimator from the MF notebook. For a cold user, map them to a sentinel row whose factors and bias stay at zero; the item rows are still available. This is the natural fallback for an ID-only model.

In [5]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_is_fitted

class BiasedMF(RegressorMixin, BaseEstimator):
    def __init__(self, n_users, n_items, K=8, lambda_=5.0,
                 n_iters=5, random_state=42):
        self.n_users = n_users
        self.n_items = n_items
        self.K = K
        self.lambda_ = lambda_
        self.n_iters = n_iters
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=int)
        y = np.asarray(y, dtype=float)
        users, items = X[:, 0], X[:, 1]

        self.mu_ = y.mean()
        self.a_ = np.zeros(self.n_users)
        self.b_ = np.zeros(self.n_items)

        rng = np.random.default_rng(self.random_state)
        self.P_ = rng.normal(0, 0.1, size=(self.n_users, self.K))
        self.Q_ = rng.normal(0, 0.1, size=(self.n_items, self.K))

        rows_by_user = [[] for _ in range(self.n_users)]
        rows_by_item = [[] for _ in range(self.n_items)]
        for row, (u, i) in enumerate(X):
            rows_by_user[u].append(row)
            rows_by_item[i].append(row)

        self.seen_users_ = np.array([bool(rows) for rows in rows_by_user])
        self.seen_items_ = np.array([bool(rows) for rows in rows_by_item])
        self.P_[~self.seen_users_] = 0.0
        self.Q_[~self.seen_items_] = 0.0

        for _ in range(self.n_iters):
            for i, rows in enumerate(rows_by_item):
                if not rows:
                    continue
                u = users[rows]
                ridge = Ridge(alpha=self.lambda_, fit_intercept=True)
                ridge.fit(self.P_[u], y[rows] - self.mu_ - self.a_[u])
                self.b_[i] = ridge.intercept_
                self.Q_[i] = ridge.coef_

            for u, rows in enumerate(rows_by_user):
                if not rows:
                    continue
                i = items[rows]
                ridge = Ridge(alpha=self.lambda_, fit_intercept=True)
                ridge.fit(self.Q_[i], y[rows] - self.mu_ - self.b_[i])
                self.a_[u] = ridge.intercept_
                self.P_[u] = ridge.coef_
        return self

    def predict(self, X):
        check_is_fitted(self, "mu_")
        X = np.asarray(X, dtype=int)
        users, items = X[:, 0], X[:, 1]
        return (self.mu_ + self.a_[users] + self.b_[items]
                + np.sum(self.P_[users] * self.Q_[items], axis=1))

def train_loop(model, data, epochs=100, lr=0.05, batch_size=256):
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    n = len(data["y"])
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            batch = {k: v[idx] for k, v in data.items() if k != "y"}
            opt.zero_grad()
            loss_fn(model(batch), data["y"][idx]).backward()
            opt.step()
    return model

def predict(model, data):
    model.eval()
    batch = {k: v for k, v in data.items() if k != "y"}
    return model(batch).detach().numpy().ravel()

In [6]:
cold_user_map = {u: k for k, u in enumerate(sorted(cold_train.user_id.unique()))}
cold_item_map = {i: k for k, i in enumerate(sorted(cold_train.dish_id.unique()))}
sentinel_user = len(cold_user_map)

X_train_mf = np.column_stack([
    cold_train.user_id.map(cold_user_map),
    cold_train.dish_id.map(cold_item_map),
]).astype(int)
X_test_mf = np.column_stack([
    cold_test.user_id.map(cold_user_map).fillna(sentinel_user),
    cold_test.dish_id.map(cold_item_map),
]).astype(int)

mf_cold = BiasedMF(
    n_users=len(cold_user_map) + 1,
    n_items=len(cold_item_map),
    K=2, lambda_=0.1, n_iters=15, random_state=42,
)
mf_cold.fit(X_train_mf, y_train_cold)
pred_mf_cold = mf_cold.predict(X_test_mf)

print("MF cold RMSE:", f"{root_mean_squared_error(y_test_cold, pred_mf_cold):.4f}")
print("MF cold MAE: ", f"{mean_absolute_error(y_test_cold, pred_mf_cold):.4f}")

MF cold RMSE: 0.9099
MF cold MAE:  0.8180


The cold-user MF is almost identical to the global mean. The item rows are learned, but there is no user representation at all. This is exactly the failure mode the lecture is about.

For reference, fit the same ALS model on the **warm** split supplied in `ratings.csv`. There every user and dish has training rows.

In [7]:
warm_train = df[df.split == "train"].copy()
warm_test = df[df.split == "test"].copy()

warm_user_map = {u: k for k, u in enumerate(sorted(warm_train.user_id.unique()))}
warm_item_map = {i: k for k, i in enumerate(sorted(warm_train.dish_id.unique()))}

print("unseen warm users:", warm_test.user_id.map(warm_user_map).isna().sum())
print("unseen warm dishes:", warm_test.dish_id.map(warm_item_map).isna().sum())

X_train_warm = np.column_stack([
    warm_train.user_id.map(warm_user_map),
    warm_train.dish_id.map(warm_item_map),
]).astype(int)
X_test_warm = np.column_stack([
    warm_test.user_id.map(warm_user_map),
    warm_test.dish_id.map(warm_item_map),
]).astype(int)

mf_warm = BiasedMF(
    n_users=len(warm_user_map),
    n_items=len(warm_item_map),
    K=2, lambda_=0.1, n_iters=15, random_state=42,
)
mf_warm.fit(X_train_warm, warm_train.rating.to_numpy())
pred_mf_warm = mf_warm.predict(X_test_warm)

print("MF warm RMSE:", f"{root_mean_squared_error(warm_test.rating, pred_mf_warm):.4f}")
print("MF warm MAE: ", f"{mean_absolute_error(warm_test.rating, pred_mf_warm):.4f}")

unseen warm users: 0
unseen warm dishes: 0


MF warm RMSE: 0.3158
MF warm MAE:  0.2502


## 6. Two-tower side-aware model

The two-tower model gives the user and the item separate encoders. The user tower uses the food preference and, when available, the user ID. The item tower uses category, canteen, continuous features, and the item ID. For a cold user, the user ID is replaced by a zero vector; the profile features still produce a representation.

In [8]:
pref_enc = LabelEncoder().fit(cold_train.food_preference)
cat_enc = LabelEncoder().fit(cold_train.category)
canteen_enc = LabelEncoder().fit(cold_train.canteen_id)

class TwoTower(nn.Module):
    def __init__(self, n_users, n_items, n_pref, n_cat, n_canteen, d_num,
                 K=16, Kc=8, hidden=32, mu=0.0):
        super().__init__()
        torch.manual_seed(0)
        self.user_emb = nn.Embedding(n_users, K, max_norm=0.3)
        self.item_emb = nn.Embedding(n_items, K, max_norm=0.3)
        self.pref_emb = nn.Embedding(n_pref, Kc, max_norm=0.3)
        self.cat_emb = nn.Embedding(n_cat, Kc, max_norm=0.3)
        self.canteen_emb = nn.Embedding(n_canteen, 4, max_norm=0.3)
        self.user_bias = nn.Parameter(torch.zeros(n_users, 1))
        self.item_bias = nn.Parameter(torch.zeros(n_items, 1))
        self.user_tower = nn.Sequential(
            nn.Linear(Kc + K, hidden), nn.ReLU(),
            nn.Linear(hidden, K),
        )
        self.item_tower = nn.Sequential(
            nn.Linear(Kc + 4 + d_num + K, hidden), nn.ReLU(),
            nn.Linear(hidden, K),
        )
        self.mu = nn.Parameter(torch.tensor([mu], dtype=torch.float32))

    def forward(self, batch):
        uid = batch["uid"]
        known = uid >= 0
        uid_safe = uid.clamp(min=0)

        hu = self.user_tower(torch.cat([
            self.pref_emb(batch["pref"]),
            self.user_emb(uid_safe) * known[:, None],
        ], dim=1))

        hi = self.item_tower(torch.cat([
            self.cat_emb(batch["cat"]),
            self.canteen_emb(batch["canteen"]),
            batch["num"],
            self.item_emb(batch["iid"]),
        ], dim=1))

        dot = (hu * hi).sum(dim=1, keepdim=True)
        ub = self.user_bias[uid_safe] * known[:, None]
        return self.mu + ub + self.item_bias[batch["iid"]] + dot

def tower_tensors(frame, user_map, item_map):
    uid = frame.user_id.map(user_map).fillna(-1).astype(int).to_numpy()
    iid = frame.dish_id.map(item_map).to_numpy()
    return {
        "uid": torch.as_tensor(uid, dtype=torch.long),
        "iid": torch.as_tensor(iid, dtype=torch.long),
        "pref": torch.as_tensor(pref_enc.transform(frame["food_preference"]), dtype=torch.long),
        "cat": torch.as_tensor(cat_enc.transform(frame["category"]), dtype=torch.long),
        "canteen": torch.as_tensor(canteen_enc.transform(frame["canteen_id"]), dtype=torch.long),
        "num": torch.as_tensor(num_scaler.transform(frame[num_item]), dtype=torch.float32),
        "y": torch.as_tensor(frame["rating"].to_numpy(dtype=np.float32).reshape(-1, 1), dtype=torch.float32),
    }

cold_train_tower = tower_tensors(cold_train, cold_user_map, cold_item_map)
cold_test_tower = tower_tensors(cold_test, cold_user_map, cold_item_map)

two_tower = TwoTower(
    n_users=len(cold_user_map),
    n_items=len(cold_item_map),
    n_pref=len(pref_enc.classes_),
    n_cat=len(cat_enc.classes_),
    n_canteen=len(canteen_enc.classes_),
    d_num=len(num_item),
    K=16, Kc=8, hidden=32,
    mu=float(y_train_cold.mean()),
)
train_loop(two_tower, cold_train_tower, epochs=100, lr=0.05)
pred_tower = predict(two_tower, cold_test_tower)

print("two-tower cold RMSE:", f"{root_mean_squared_error(y_test_cold, pred_tower):.4f}")
print("two-tower cold MAE: ", f"{mean_absolute_error(y_test_cold, pred_tower):.4f}")

two-tower cold RMSE: 0.3456
two-tower cold MAE:  0.2747


In [9]:
comparison = pd.DataFrame([
    {"model": "global mean", "features": "none",
     "cold RMSE": root_mean_squared_error(y_test_cold, np.full_like(y_test_cold, y_train_cold.mean())),
     "cold MAE": mean_absolute_error(y_test_cold, np.full_like(y_test_cold, y_train_cold.mean()))},
    {"model": "feature-only ridge", "features": "continuous + categorical",
     "cold RMSE": root_mean_squared_error(y_test_cold, pred_ridge),
     "cold MAE": mean_absolute_error(y_test_cold, pred_ridge)},
    {"model": "ID-only MF", "features": "IDs only",
     "cold RMSE": root_mean_squared_error(y_test_cold, pred_mf_cold),
     "cold MAE": mean_absolute_error(y_test_cold, pred_mf_cold)},
    {"model": "two-tower", "features": "IDs + continuous + categorical",
     "cold RMSE": root_mean_squared_error(y_test_cold, pred_tower),
     "cold MAE": mean_absolute_error(y_test_cold, pred_tower)},
]).set_index("model")

display(comparison.round(4))
print("warm MF reference RMSE:", f"{root_mean_squared_error(warm_test.rating, pred_mf_warm):.4f}")

,features,cold RMSE,cold MAE
model,,,
global mean,none,0.9113,0.8158
feature-only ridge,continuous + categorical,0.7558,0.5997
ID-only MF,IDs only,0.9099,0.8180
two-tower,IDs + continuous + categorical,0.3456,0.2747


warm MF reference RMSE: 0.3158


The pattern is the one from the slides:

- **warm MF** is strong because every user and dish has a learned ID row;
- **cold MF** collapses to the global mean;
- **feature-only** recovers a large part of the signal;
- **two-tower** learns the interaction between user preferences and dish features, and improves the cold segment further.

The canteen data are synthetic and have a strong latent group structure. The relative ordering is the lesson, not the exact RMSE.

## 7. Optional: text as another encoder

The dish name is short, but it is still text. We use character n-grams + SVD as a **stand-in** for an LLM embedding. In production, replace this block with a sentence-transformer or API embedding; the interface is the same: `text -> fixed vector`.

This is optional. The required part of the lecture ends with the two-tower comparison above.

In [10]:
text_vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(1, 2), max_features=500)
text_vectorizer.fit(cold_train.dish.unique())

text_svd = TruncatedSVD(n_components=16, random_state=0)
text_svd.fit(text_vectorizer.transform(cold_train.dish.unique()))

X_train_text = text_svd.transform(text_vectorizer.transform(cold_train.dish))
X_test_text = text_svd.transform(text_vectorizer.transform(cold_test.dish))

ridge_text = Ridge(alpha=1.0).fit(
    np.hstack([X_train_side, X_train_text]), y_train_cold
)
pred_text = ridge_text.predict(np.hstack([X_test_side, X_test_text]))

print("feature-only + text RMSE:", f"{root_mean_squared_error(y_test_cold, pred_text):.4f}")
print("feature-only + text MAE: ", f"{mean_absolute_error(y_test_cold, pred_text):.4f}")

feature-only + text RMSE: 0.7546
feature-only + text MAE:  0.5988


Text is not automatically helpful. The gain depends on whether the text contains information that the other features miss. Also check availability and leakage: a review written after the rating may mention the rating itself.

Image side information follows the same pattern: a pretrained CNN or ViT produces a vector, and the recommender fuses that vector. The canteen dataset has only eight canteen photos, so we do not build a per-dish image model here; the lecture treats image as an optional encoder.

## 8. Exercises

1. **Add a feature.** Add a price bucket or a missing-value flag. Does cold-user RMSE improve?
2. **Change the encoder.** Replace the category embedding with a one-hot linear model. What changes?
3. **Cold items.** Hold out 20% of dishes instead of users. Does the two-tower still help?
4. **Audit leakage.** Add a rating-mean feature computed globally, then recompute it inside the fold. Compare.
5. **Optional text.** Replace the TF-IDF block with a precomputed LLM embedding and rerun the comparison.
6. **Optional image.** Use the eight canteen photos as a canteen-level image feature and test whether it adds signal.

For each exercise, keep the split fixed and change one thing at a time.

## Summary

```text
ratings + user features + item features
  → cold-start split
  → feature-only baseline
  → ID-only MF with zero fallback
  → two-tower with continuous + categorical features
  → warm vs cold comparison
  → optional text / image encoder
```

The model is only as good as the split. Report warm and cold metrics separately, audit every rating-derived feature, and measure whether the optional encoder earns its cost.